####  20 — Final Comparison and Conclusion

##### 1. Notebook purpose


#####  Objective

Consolidate the results and lessons from the Product Defect Computer
Vision project.

This notebook summarizes the complete journey:

    image data
        ↓
    image representation and preprocessing
        ↓
    custom CNN
        ↓
    transfer learning with ResNet18
        ↓
    fine-tuning
        ↓
    model evaluation
        ↓
    Explainable AI (XAI)
        ↓
    Responsible AI (RAI)
        ↓
    MLflow tracking
        ↓
    Unity Catalog model registration
        ↓
    Databricks Model Serving
        ↓
    endpoint testing

No new model is trained in this notebook.

The goal is to compare the approaches, document the final system,
summarize limitations, and capture the main technical lessons.

##### 2. Project problem

##### Business Problem

The objective of this project was to classify casting-product images
into two categories:

- `def_front` — defective product
- `ok_front` — acceptable product

This is a binary image-classification problem.

The project was designed not only to train a classifier, but also to
practice the complete lifecycle of a production-oriented computer
vision system.

The basic ML problem was:

``` text

Casting image
      ↓
Computer Vision Model
      ↓
def_front OR ok_front

```

But the project eventually became:

``` text

Data
 ↓
Model development
 ↓
Evaluation
 ↓
Explainability
 ↓
Responsible AI assessment
 ↓
Experiment tracking
 ↓
Governed registration
 ↓
Production inference packaging
 ↓
Deployment
 ↓
Endpoint validation

```

That difference is important.

##### 3. Dataset summary


The project used the Casting Product Image Data for Quality Inspection
dataset.

The original directory structure contained:

Development data:
- def_front: 3,758 images
- ok_front: 2,875 images

Test data:
- def_front: 453 images
- ok_front: 262 images

Total:
- 7,348 images

The original development data was split into training and validation
subsets while preserving the original test directory as the final test
set.

##### 4. Image representation


Images were converted into numerical tensors so they could be processed
by neural networks.

A typical RGB image became:     [3, 224, 224]

where:

    3   = RGB channels
    224 = image height
    224 = image width

During training, DataLoader added the batch dimension:     [B, 3, 224, 224]

For example:     [32, 3, 224, 224]

represents a batch of 32 RGB images.

##### 5. Custom CNN

In [0]:

The first model was built from scratch to understand how convolutional
neural networks learn image features.

Architecture:

``` text

Input
    ↓
Conv2d 3 → 32
ReLU
MaxPool
    ↓
Conv2d 32 → 64
ReLU
MaxPool
    ↓
Conv2d 64 → 128
ReLU
MaxPool
    ↓
Adaptive Average Pooling
    ↓
Linear 128 → 64
ReLU
    ↓
Linear 64 → 2
    ↓
Logits

```

##### 6. Custom CNN results


Final test results:

- Accuracy: 83.22%
- Errors: 120 / 715
- Defect recall: 84.99%

Confusion matrix:

``` text

                 Predicted
               Defect    OK

Actual Defect    385      68
Actual OK         52     210

```

The custom CNN was valuable because it taught us how CNNs work, even though it was not the strongest classifier.
That's an important distinction:
A model can be extremely valuable for learning and experimentation without being the model we ultimately deploy.

##### 7. Why transfer learning?

In [0]:

Instead of learning every visual feature from scratch, the second
approach used a pretrained ResNet18 model.

ResNet18 had already learned useful visual representations from a much
larger image dataset.

The final classification layer was replaced for the project's two
classes:

``` text

    pretrained visual feature extractor
                ↓
             512 features
                ↓
          Linear(512 → 2)
                ↓
             logits

```

This allowed us to reuse general visual knowledge and adapt it to product-defect classification.

##### 8. Feature extraction and fine-tuning

We used two stages.

**STAGE 1 — FEATURE EXTRACTION**

``` text

Pretrained ResNet18
most layers frozen
       ↓
train new classifier head

```

Then:

**STAGE 2 — PARTIAL FINE-TUNING**

``` text

earlier pretrained layers frozen
       ↓
layer4 trainable
       ↓
fc trainable
       ↓
adapt deeper features to casting defects

```

The fine-tuning learning rate was smaller than the feature-extraction
learning rate because pretrained parameters were already useful and
should be adjusted carefully rather than changed aggressively.

##### 9. Final model comparison

| Metric | Custom CNN | Fine-tuned ResNet18 |
|---|---:|---:|
| Test accuracy | 83.22% | **99.58%** |
| Test errors | 120 | **3** |
| Defect correctly detected | 385 / 453 | **450 / 453** |
| Defect false negatives | 68 | **3** |
| Defect recall | 84.99% | **99.34%** |

##### ResNet18 confusion matrix:
                 Predicted
               Defect    OK

Actual Defect    450       3
Actual OK          0     262

The evidence from this test set therefore strongly favored the fine-tuned ResNet18 for this project.

But we should phrase that carefully:
- The fine-tuned ResNet18 substantially outperformed the custom CNN on this project's test data.

- This result supports using ResNet18 as the final project model, but it
does not establish that ResNet18 will always outperform custom CNNs on
other datasets or production environments.

##### 10. Why did ResNet18 perform better?


The custom CNN learned visual representations only from the project's
relatively small training dataset.

ResNet18 started with visual representations learned from a much larger
pretraining dataset.

Fine-tuning then adapted some of those pretrained representations to
the casting-defect task.

Conceptually:

Custom CNN:

``` text

    random initialization
          ↓
    project images only
          ↓
    learn visual features
          ↓
    classify

```

Transfer learning:

``` text

    pretrained visual knowledge
          ↓
    project images
          ↓
    adapt useful features
          ↓
    classify

```

##### 11. Evaluation integrity — duplicate analysis


Exact SHA-256 comparison identified 64 exact duplicate images between
the original development and test directories.

That represents approximately:

    64 / 715 = 8.95%

of the original test set.

Because overlap between development and test data can make evaluation
results appear more favorable, the models were also evaluated on the
non-duplicate test subset.

| Model | Original test | Non-duplicate test subset |
|---|---:|---:|
| Custom CNN | 83.22% | 83.26% |
| ResNet18 | 99.58% | 99.54% |


ResNet18 non-duplicate confusion matrix:
[[450, 3],
 [  0,198]]

This is useful evidence because performance remained strong after removing the known exact duplicates.

But:

Exact hash matching detects exact duplicates only.

It does not prove the absence of:

- near duplicates
- sequential images
- images of the same physical product
- correlated manufacturing samples

Therefore we call it the non-duplicate test subset, not a “clean test set.”

##### 12. The three remaining ResNet18 errors

| Image | Actual | Predicted | Approx. confidence |
|---|---|---|---:|
| `cast_def_0_150.jpeg` | def_front | ok_front | 98.46% |
| `cast_def_0_1591.jpeg` | def_front | ok_front | 67.55% |
| `cast_def_0_2102.jpeg` | def_front | ok_front | 85.60% |

These became particularly valuable because they connected model evaluation → XAI → RAI → endpoint testing.

##### 13. 🔍 Explainable AI


Grad-CAM was used to investigate which spatial regions influenced
selected ResNet18 predictions.

The target convolutional layer was:     model.layer4[-1].conv2

For one image:

    activation grids:
    [1, 512, 7, 7]

    gradient grids:
    [1, 512, 7, 7]

Grad-CAM combined information across these channels to create a
class-specific spatial heatmap.


``` text

Image
  ↓
ResNet18 prediction
  ↓
target class
  ↓
activations + gradients
  ↓
Grad-CAM heatmap
  ↓
visual indication of influential regions

```

**What Grad-CAM does not prove**

- A Grad-CAM heatmap should not automatically be interpreted as verified
defect localization.
- It indicates spatial regions that influenced the selected class
prediction.
- The heatmap is also relatively coarse because the selected deeper
convolutional layer has a 7 × 7 spatial activation grid.


That distinction is essential:

**Grad-CAM:**
"Where was the model relatively responsive?"

**NOT necessarily:**
"This is exactly where the physical defect is."

##### 14. ⚖️ Responsible AI


Responsible AI analysis considered whether the model results could be
appropriately trusted and used.

Important findings included:

##### Reliability and safety

Three defect images were classified as OK.

For quality inspection, false negatives may have meaningful operational
consequences.

##### Confidence

One incorrect prediction had approximately 98.46% confidence.

Therefore:

    confidence ≠ correctness

##### Evaluation integrity

Exact duplicate images were found between development and test data.

The non-duplicate subset was evaluated separately.

##### Robustness

The model has not been established as robust across changes such as:

- camera hardware
- lighting
- equipment
- product batches
- image positioning
- manufacturing environments

##### Human oversight

The project does not prescribe a human-review threshold.

Such a threshold should depend on real operational costs, safety
requirements, inspection capacity, and acceptable risk.


**A useful summary:**

- 🔍 XAI - Why might the model have predicted this?
- ⚖️ RAI - Can we appropriately trust and use this prediction
for the intended situation?

##### 15. MLflow experiment tracking


- MLflow was used to persist experiment information and model artifacts.

- This allowed model development to move beyond notebook memory.

- For the final ResNet18 model, MLflow preserved the trained model
artifact and associated experiment information.

##### 16. Model registration and governance


- The final model was registered in Unity Catalog as: dbw_agentic_ai_dev.product_defect_ai.product_defect_classifier

- Registration gave the model a managed identity with versioning, metadata, lineage, and access-control capabilities.

Governed = managed and controlled in an organized, traceable way.

Lineage:

``` text

Training Run
3f9b44...
     ↓
Core Model
Version 1
     ↓
Serving Package
Version 2
     ↓
Corrected Serving Package
Version 3


And:
Governance
   ├── identity
   ├── versions
   ├── lineage
   ├── permissions
   └── traceability

```

##### 17. Core model vs serving model


This was one of our biggest deployment concepts.

CORE MODEL — Version 1

``` text

Tensor [B,3,224,224]
        ↓
     ResNet18
        ↓
   logits [B,2]

```

But a real application has an image, not a prepared tensor.

So the serving interface became:

``` text

SERVING MODEL

Base64 image
     ↓
decode
     ↓
RGB image
     ↓
preprocessing
     ↓
tensor
     ↓
ResNet18
     ↓
logits
     ↓
softmax
     ↓
class mapping
     ↓
structured prediction

```

The serving package did not train another classifier.

It wrapped the same learned ResNet18 behavior in an application-friendly
inference interface.

##### 18. Deployment incident and MLOps lesson


The first serving deployment failed during container dependency
resolution.

The notebook environment reported:

    torch==2.7.0+cpu
    torchvision==0.22.0+cpu

Those local build identifiers were copied into the serving package.

The serving environment could resolve:

    torch==2.7.0
    torchvision==0.22.0

but not the exact `+cpu` package specifications.

The dependency versions were normalized:

    2.7.0+cpu → 2.7.0
    0.22.0+cpu → 0.22.0

The serving package was registered as a new version and the existing
endpoint was updated.

No model retraining was required.

This gave us a very important lesson:

DEPLOYMENT FAILURE does not automatically mean MODEL FAILURE

The problem was: environment / dependency packaging

not:
- training
- weights
- model architecture
- Base64 inference logic

##### 19. Why Version 3 did not mean retraining


A new registered model version does not necessarily mean that a model
was retrained.

Version 3 used the same learned ResNet18 weights.

The change was to the serving-package dependency specification.

Model versions can therefore represent changes to:

- learned weights
- inference code
- preprocessing
- dependencies
- interface
- packaging
- metadata

Versioning tracks the deployable artifact, not only training events.

##### 20. Final deployment

Endpoint: product-defect-classifier-endpoint

Serving model:
- product_defect_classifier
- Version 3

Compute:
CPU
Small
Scale to zero enabled

Final inference:

``` text

Client
  ↓
Base64 image
  ↓
Databricks endpoint
  ↓
MLflow signature
  ↓
serving wrapper
  ↓
ResNet18 preprocessing
  ↓
ResNet18
  ↓
prediction

```

##### 21. Endpoint testing

| Test | Result |
|---|---|
| Known defect | PASS |
| Known OK | PASS |
| Response schema | PASS |
| Probability consistency | PASS |
| Known false-negative behavior | PASS |
| Batch inference | PASS |
| Missing `image_base64` | PASS |
| Invalid Base64 | PASS |
| Empty Base64 | PASS |

The known false-negative test passing means:
Endpoint behavior matches evaluated model behavior

It does not mean:
The false-negative prediction is correct.

It remains a model error.

##### 22. Why the MLflow signature mattered


The deployed model signature required:     image_base64: string

When endpoint testing sent:     wrong_column

the request was rejected because the required input was missing.

Therefore, the model signature was not merely documentation; it
participated in enforcing the serving interface.

##### 23. Complete project architecture


``` text

                         DATA
                          │
                          ▼
              Casting product images
                          │
                          ▼
             Train / Validation / Test
                          │
            ┌─────────────┴─────────────┐
            ▼                           ▼
       Custom CNN                 Transfer Learning
                                      ResNet18
            │                           │
            └─────────────┬─────────────┘
                          ▼
                     Evaluation
                          │
                          ├───────────────┐
                          ▼               ▼
                      🔍 XAI            ⚖️ RAI
                     Grad-CAM       Reliability
                                    Data integrity
                                    Limitations
                          │               │
                          └───────┬───────┘
                                  ▼
                            MLflow Tracking
                                  │
                                  ▼
                         Unity Catalog Registry
                                  │
                                  ▼
                         Serving Model Package
                                  │
                                  ▼
                         Databricks Model Serving
                                  │
                                  ▼
                         Endpoint Testing
                                  │
                                  ▼
                         Tested AI System

```

##### 24. Technologies practiced

##### Technologies and Concepts Practiced

##### Computer Vision
- pixels and RGB channels
- image tensors
- resizing and normalization
- image augmentation
- convolution
- pooling
- CNN feature learning

##### Deep Learning
- PyTorch
- forward propagation
- loss
- backpropagation
- Adam optimization
- epochs
- logits
- softmax

##### Transfer Learning
- pretrained ResNet18
- feature extraction
- freezing parameters
- partial fine-tuning

##### Evaluation
- accuracy
- confusion matrix
- recall
- false positives
- false negatives
- duplicate-data auditing

##### 🔍 Explainable AI
- activation grids
- gradient grids
- Grad-CAM
- explanation limitations

##### ⚖️ Responsible AI
- reliability
- safety
- evaluation integrity
- robustness
- transparency
- human oversight
- appropriate-use limitations

##### MLOps
- MLflow experiments
- logged models
- model signatures
- input examples
- Unity Catalog model registry
- model versions
- lineage
- governance
- dependency packaging

##### Deployment
- MLflow PyFunc
- Base64 image interface
- Databricks Model Serving
- endpoint updates
- container dependency troubleshooting

##### Production Testing
- endpoint invocation
- response-schema testing
- probability validation
- batch requests
- negative testing
- malformed-input handling

##### 25. Most important conceptual lessons

##### Key Learnings

1. Images must be represented numerically before a neural network can
   process them.

2. CNNs learn spatial visual patterns through convolutional filters.

3. Deeper CNN layers learn increasingly task-relevant representations.

4. Transfer learning can reuse visual representations learned from much
   larger datasets.

5. Fine-tuning adapts pretrained representations to the target task.

6. A higher test score should still be examined for data leakage,
   duplicates, error types, and robustness limitations.

7. False negatives can matter differently from false positives depending
   on the operational problem.

8. Confidence is not the same as correctness.

9. 🔍 XAI investigates why a model produced a prediction.

10. Grad-CAM indicates influential spatial regions but does not prove
    physical defect localization.

11. ⚖️ RAI evaluates whether and how model predictions can be
    appropriately trusted and used.

12. Experiment tracking preserves model-development evidence beyond
    notebook memory.

13. Lineage tells us where a model came from.

14. Governance provides organized management, control, versioning,
    permissions, and traceability.

15. A trained model is only one component of an inference system.

16. A production inference pipeline includes preprocessing, model
    execution, postprocessing, and an application-facing interface.

17. Model signatures help define and enforce serving contracts.

18. A deployment failure can originate from environment packaging rather
    than model training.

19. A new model version does not necessarily mean model retraining.

20. Endpoint testing and model evaluation answer different questions.

21. Positive and negative endpoint tests are both important.

22. A READY endpoint means the service deployed successfully; it does
    not by itself establish that the model is safe or robust for every
    production environment.

##### 26. How to explain this project in an interview

##### Summary

- I built an end-to-end computer vision system for detecting defects in casting-product images using Databricks and PyTorch.

- I first built a custom CNN to understand convolution, pooling, feature learning, and image classification from first principles.

- I then implemented transfer learning using pretrained ResNet18. I first used it as a feature extractor and then partially fine-tuned the deeper
layer and classifier.

- On the project test set, the custom CNN achieved approximately 83.22% accuracy, while the fine-tuned ResNet18 achieved approximately 99.58%.

- I also audited the evaluation data and found exact overlap between the development and test directories. After removing known exact duplicates,
ResNet18 still achieved approximately 99.54% accuracy on the non-duplicate test subset.

- I investigated model behavior with Grad-CAM and assessed Responsible AI considerations such as false-negative risk, confidence calibration
limitations, dataset integrity, distribution shift, and human oversight.

- I tracked the model with MLflow, registered it in Unity Catalog with lineage metadata, and created a serving-oriented MLflow PyFunc interface
that accepts Base64 images.

- During deployment, I diagnosed a container dependency failure caused by notebook-specific PyTorch `+cpu` build metadata. I corrected the serving
environment without retraining the model, registered a new governed model version, and updated the existing Databricks serving endpoint.

- Finally, I tested the live endpoint for successful predictions, response schema, probability consistency, batch inference, known model errors,
and malformed inputs.

- This project gave me experience across the complete computer vision lifecycle: model development, evaluation, explainability, Responsible
AI, MLOps, deployment, troubleshooting, and production endpoint testing.

##### 27. Final conclusion


- This project progressed from understanding individual image pixels to building and validating a complete deployed computer vision system.

- The custom CNN provided the foundation for understanding how neural networks learn visual features.

- Transfer learning with ResNet18 demonstrated the value of pretrained representations and substantially improved classification performance on this dataset.

- Model evaluation identified the remaining false-negative cases, while duplicate-data analysis highlighted the importance of evaluation integrity.

- Explainable AI using Grad-CAM provided additional visibility into model behavior without being treated as verified defect localization.

- Responsible AI analysis prevented strong test performance from being interpreted as proof of universal production reliability.

- MLflow provided experiment and model persistence, while Unity Catalog provided governed model identity, versioning, lineage, and traceability.

- The deployment phase demonstrated that production ML involves more than model weights. Inference code, preprocessing, dependencies, signatures,
interfaces, and serving infrastructure must all work together.

- A real deployment failure caused by dependency packaging was diagnosed and corrected without retraining the classifier, demonstrating the
difference between model failures and serving-environment failures.

- Finally, the live Databricks endpoint was validated through both successful and intentionally invalid requests.

The final result is not simply a trained image classifier.

It is an end-to-end computer vision system covering:

    data
      → model development
      → evaluation
      → XAI
      → RAI
      → experiment tracking
      → governance
      → deployment
      → endpoint validation

This completes the Product Defect Computer Vision learning project.